# MTH522 Project 2: Civil Unrest and Violence Trends in India

This notebook is reconstructed from the PDF report and includes the exact extracted visuals from the original file.

- Source PDF: `MTH522_Report(Project-2) (2).pdf`
- Expected dataset path: `data/ACLED-India (1) (1).xlsx`
- Expected sheet name: `1900-01-01-2025-01-10-India`


## Dataset Note

The original report code points to an ACLED Excel workbook that is not present in this workspace yet. The notebook is prepared so you can drop the file into `data/` and run the analysis. Until then, the exact report images are shown below from the extracted PDF assets.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

DATA_PATH = Path("data/ACLED-India (1) (1).xlsx")
SHEET_NAME = "1900-01-01-2025-01-10-India"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Missing dataset: {DATA_PATH}. Place the ACLED Excel file there, then rerun the notebook."
    )

df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
df.columns = [str(col).strip() for col in df.columns]

sparse_columns = ["assoc_actor_2", "civilian_targeting", "tags", "timestamp"]
df_cleaned = df.drop(columns=[c for c in sparse_columns if c in df.columns], errors="ignore")

if "event_date" in df_cleaned.columns:
    df_cleaned["event_date"] = pd.to_datetime(df_cleaned["event_date"], errors="coerce")
    df_cleaned["year"] = df_cleaned["event_date"].dt.year

violent_event_types = [
    "Battles",
    "Violence against civilians",
    "Riots",
    "Explosions/Remote violence",
]

violent_events = df_cleaned[df_cleaned["event_type"].isin(violent_event_types)].copy()
protests_events = df_cleaned[df_cleaned["event_type"] == "Protests"].copy()

print("Rows loaded:", len(df_cleaned))
print("Protest events:", len(protests_events))
print("Violent events:", len(violent_events))


## Original Loading / Filtering Code Screenshot

![Original load code](extracted_pdf_images/page_19_img_01.png)

In [ ]:
# Top States with the highest number of Protests
top_states_protests = protests_events["admin1"].value_counts().head(10)
print(top_states_protests)

# Top Cities with the highest number of Protests
city_col = "admin2" if "admin2" in protests_events.columns else "location"
top_cities_protests = protests_events[city_col].value_counts().head(10)
print(top_cities_protests)


## Exact Report Images: Protest Concentration

![Top protest states](extracted_pdf_images/page_08_img_01.png)

![Top protest cities](extracted_pdf_images/page_09_img_01.png)

![Original protest-count code](extracted_pdf_images/page_20_img_02.png)

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(protests_events["longitude"], protests_events["latitude"], alpha=0.5, s=10, color="blue")
plt.title("Scatter Map of Protest Events (India, 2022-2024)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
plt.scatter(violent_events["longitude"], violent_events["latitude"], alpha=0.5, s=10, color="red")
plt.title("Scatter Map of Violent Events (India, 2022-2024)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True)
plt.tight_layout()
plt.show()


## Exact Report Images: Maps

![Protest map](extracted_pdf_images/page_10_img_01.png)

![Violence map](extracted_pdf_images/page_11_img_01.png)

![Original protest map code](extracted_pdf_images/page_20_img_01.png)

![Original violent map code](extracted_pdf_images/page_20_img_03.png)

In [ ]:
protest_notes_text = " ".join(protests_events["notes"].dropna().astype(str).str.lower().tolist())
wordcloud = WordCloud(width=800, height=400, background_color="white", max_words=100).generate(protest_notes_text)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation="bilinear")
plt.axis("off")
plt.title("Key Reasons Behind Protests (Word Cloud)", fontsize=16)
plt.tight_layout()
plt.show()


## Exact Report Image: Protest Reasons

![Word cloud](extracted_pdf_images/page_12_img_01.png)

![Original word-cloud code](extracted_pdf_images/page_20_img_04.png)

In [ ]:
# Top States with highest number of Violent Events
top_states_violence = violent_events["admin1"].value_counts().head(10)
print(top_states_violence)

violent_events_filtered = violent_events[(violent_events["year"] >= 2015) & (violent_events["year"] <= 2025)]
violent_events_by_year = violent_events_filtered["year"].value_counts().sort_index()
violent_events_full = pd.Series(index=range(2015, 2026), dtype=int)
for year, count in violent_events_by_year.items():
    violent_events_full[year] = count
violent_events_full.fillna(0, inplace=True)

plt.figure(figsize=(10, 6))
violent_events_full.plot(marker="o", linestyle="-")
plt.title("Violent Events in India (2015-2025)")
plt.xlabel("Year")
plt.ylabel("Number of Violent Events")
plt.grid(True)
plt.tight_layout()
plt.show()

protests_events_filtered = protests_events[(protests_events["year"] >= 2015) & (protests_events["year"] <= 2025)]
protests_by_year = protests_events_filtered["year"].value_counts().sort_index()
protests_full = pd.Series(index=range(2015, 2026), dtype=int)
for year, count in protests_by_year.items():
    protests_full[year] = count
protests_full.fillna(0, inplace=True)

plt.figure(figsize=(10, 6))
protests_full.plot(marker="o", linestyle="-", color="blue")
plt.title("Protest Events in India (2015-2025)")
plt.xlabel("Year")
plt.ylabel("Number of Protest Events")
plt.grid(True)
plt.tight_layout()
plt.show()


## Exact Report Images: Violence and Time Series

![Violent states](extracted_pdf_images/page_13_img_01.png)

![Violence by year](extracted_pdf_images/page_14_img_01.png)

![Protests by year](extracted_pdf_images/page_15_img_01.png)

![Original violent-state code](extracted_pdf_images/page_21_img_01.png)

![Original violent-series code](extracted_pdf_images/page_21_img_02.png)

![Original protest-series code](extracted_pdf_images/page_21_img_03.png)

In [ ]:
def assign_group(row):
    actor_info = str(row["actor1"]).lower()
    note_info = str(row["notes"]).lower()

    if "student" in note_info or "youth" in note_info:
        return "Student Groups"
    elif "farmer" in note_info or "agriculture" in note_info:
        return "Farmers' Associations"
    elif "worker" in note_info or "labor" in note_info or "union" in note_info:
        return "Labor Unions"
    elif "religious" in note_info or "temple" in note_info or "mosque" in note_info:
        return "Religious Organizations"
    elif "tribal" in note_info or "ethnic" in note_info or "indigenous" in note_info:
        return "Ethnic/Tribal Groups"
    elif "police" in actor_info or "military" in actor_info or "forces" in actor_info or "security" in actor_info:
        return "Security Forces"
    elif "party" in actor_info or "political" in note_info or "bjp" in note_info or "congress" in note_info:
        return "Political Organizations"
    else:
        return "Others"

protests_events_filtered["actor_group"] = protests_events_filtered.apply(assign_group, axis=1)
actor_group_counts = protests_events_filtered["actor_group"].value_counts()
print(actor_group_counts)

plt.figure(figsize=(10, 6))
actor_group_counts.sort_values(ascending=True).plot(kind="barh", color="purple")
plt.title("Protest Events by Actor Group (2015-2025)")
plt.xlabel("Number of Protest Events")
plt.ylabel("Actor Group")
plt.grid(axis="x", linestyle="--")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 8))
actor_group_counts.plot(kind="pie", autopct="%1.1f%%", startangle=140, colors=plt.cm.Set3.colors)
plt.title("Protest Events by Actor Group (2015-2025)")
plt.ylabel("")
plt.tight_layout()
plt.show()


## Exact Report Images: Actor Grouping

![Actor grouping chart](extracted_pdf_images/page_17_img_01.png)

![Actor grouping pie chart](extracted_pdf_images/page_18_img_01.png)

![Original actor-grouping code](extracted_pdf_images/page_22_img_01.png)

![Original actor-chart code](extracted_pdf_images/page_22_img_02.png)